# Chapter 1: Probability Foundations
### Bayesian Analysis in Astrophysics

This notebook builds the mathematical foundations of probability theory you need before touching Bayes.
We cover sample spaces, the three Kolmogorov axioms, key distributions used in astronomy, and the Central Limit Theorem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import factorial

# Nice plotting defaults
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
print("Libraries loaded.")


## 1.1 The Poisson Distribution — Photon Counting

In X-ray and gamma-ray astronomy, detectors count individual photons.
If a source has a mean count rate λ photons/s, the number of counts k in 1 second follows:

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

Key properties: **mean = variance = λ** — so at low counts, Poisson noise dominates.


In [ ]:
# Poisson PMF for several count rates
lambdas = [1.0, 3.0, 8.0, 20.0]
k_max   = 40
k       = np.arange(0, k_max + 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
colors = ["#3B8BD4", "#1D9E75", "#D85A30", "#7F77DD"]

for ax, lam, col in zip(axes, lambdas, colors):
    pmf = stats.poisson.pmf(k, lam)
    ax.bar(k[:25], pmf[:25], color=col, alpha=0.85, width=0.7)
    ax.axvline(lam, color="black", lw=1.5, ls="--", label=f"λ={lam}")
    ax.set_xlabel("Counts k")
    ax.set_ylabel("P(X = k)")
    ax.set_title(f"Poisson(λ={lam})")
    ax.legend(frameon=False)

plt.suptitle("Poisson Distribution — Photon Counting in Astronomy", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


**Observation:** As λ grows, the Poisson distribution becomes increasingly symmetric and bell-shaped.
This is the CLT at work — we'll prove it at the end of this notebook.

> **Exercise 1.1:** For a 0.3–10 keV X-ray source with mean rate λ = 3.7 counts/s,
> compute P(X=0), P(X≤2), and P(X≥5). Interpret these physically.


In [ ]:
# Exercise 1.1 — Poisson probabilities for an X-ray source
lam = 3.7

p_zero   = stats.poisson.pmf(0, lam)
p_leq2   = stats.poisson.cdf(2, lam)
p_geq5   = 1 - stats.poisson.cdf(4, lam)

print(f"λ = {lam} counts/s")
print(f"P(X = 0)  = {p_zero:.4f}  ({p_zero*100:.2f}%)  ← source appears silent")
print(f"P(X ≤ 2)  = {p_leq2:.4f}  ({p_leq2*100:.2f}%)  ← fewer than 3 counts")
print(f"P(X ≥ 5)  = {p_geq5:.4f}  ({p_geq5*100:.2f}%)  ← burst detection threshold")


## 1.2 The Gaussian Distribution — Measurement Noise

Most spectroscopic and photometric measurements at moderate-to-high counts
are modelled as Gaussian noise:

$$f(x \mid \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

The **68-95-99.7 rule**: 68%, 95%, 99.7% of probability mass lies within 1σ, 2σ, 3σ of the mean.


In [ ]:
# Gaussian distribution and the σ-interval rule
fig, ax = plt.subplots(figsize=(10, 5))

mu, sigma = 100.0, 15.0        # e.g. flux in mJy, uncertainty 15 mJy
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
pdf = stats.norm.pdf(x, mu, sigma)

ax.plot(x, pdf, "#3B8BD4", lw=2.5, label=f"N(μ={mu}, σ={sigma})")

# Fill σ intervals
fills = [(1, "#3B8BD4", 0.35, "±1σ  (68.3%)"),
         (2, "#1D9E75", 0.20, "±2σ  (95.4%)"),
         (3, "#D85A30", 0.12, "±3σ  (99.7%)")]

for nsig, col, alpha, label in fills:
    mask = np.abs(x - mu) <= nsig * sigma
    ax.fill_between(x, pdf, where=mask, color=col, alpha=alpha, label=label)

ax.axvline(mu, color="black", lw=1.2, ls="--", alpha=0.5)
ax.set_xlabel("Flux (mJy)")
ax.set_ylabel("Probability density")
ax.set_title("Gaussian Noise Model — Spectroscopic Flux Measurement")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

# Numerical check
for nsig in [1, 2, 3]:
    frac = stats.norm.cdf(nsig) - stats.norm.cdf(-nsig)
    print(f"  ±{nsig}σ contains {frac*100:.2f}% of probability")


## 1.3 The Cauchy / Lorentzian Distribution — Spectral Lines

Atomic emission and absorption lines have a **natural line profile** (Lorentzian)
from the Heisenberg uncertainty principle. It has **heavier tails** than a Gaussian:

$$f(x \mid x_0, \gamma) = \frac{1}{\pi\gamma\left[1 + \left(\frac{x - x_0}{\gamma}\right)^2\right]}$$

Unlike the Gaussian, the Cauchy distribution has **no finite mean or variance** — outliers matter!


In [ ]:
# Gaussian vs Cauchy (Lorentzian): comparing tails
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(-10, 10, 1000)
gauss  = stats.norm.pdf(x, 0, 1)
cauchy = stats.cauchy.pdf(x, 0, 1)

# Full view
ax1.plot(x, gauss,  "#3B8BD4", lw=2.5, label="Gaussian N(0,1)")
ax1.plot(x, cauchy, "#D85A30", lw=2.5, label="Cauchy (Lorentzian)")
ax1.set_xlabel("Wavelength offset (σ units)")
ax1.set_ylabel("Probability density")
ax1.set_title("Full distribution")
ax1.legend(frameon=False)

# Tail region — log scale
ax2.semilogy(x, gauss,  "#3B8BD4", lw=2.5, label="Gaussian")
ax2.semilogy(x, cauchy, "#D85A30", lw=2.5, label="Cauchy (Lorentzian)")
ax2.set_xlim(2, 10)
ax2.set_xlabel("Wavelength offset (σ units)")
ax2.set_ylabel("log Probability density")
ax2.set_title("Tail region (log scale)")
ax2.legend(frameon=False)

plt.suptitle("Gaussian vs Lorentzian — Natural Line Profile Has Heavy Tails", fontsize=12)
plt.tight_layout()
plt.show()

print("At x=5σ:")
print(f"  Gaussian:  {stats.norm.pdf(5):.2e}")
print(f"  Cauchy:    {stats.cauchy.pdf(5):.2e}")
print(f"  Ratio:     {stats.cauchy.pdf(5)/stats.norm.pdf(5):.0f}× more likely in Cauchy")


## 1.4 Marginalisation — Integrating Out Unknowns

One of the most powerful operations in Bayesian analysis: if we have a joint distribution
f(x, y) but only care about x, we **marginalise** over y:

$$f_X(x) = \int_{-\infty}^{\infty} f(x, y)\, dy$$

**Astrophysical use:** A galaxy spectrum depends on both stellar age τ and metallicity Z.
If we only care about the age, we integrate over all possible metallicities.


In [ ]:
# Visualise marginalisation: 2D Gaussian → 1D marginals
from matplotlib.patches import Ellipse

# Correlated 2D Gaussian (like stellar age vs metallicity)
mu_vec   = np.array([8.0, -0.3])   # [log age (Gyr), [Fe/H]]
cov      = np.array([[0.4, 0.15], [0.15, 0.08]])

rv = stats.multivariate_normal(mean=mu_vec, cov=cov)

age_grid = np.linspace(6.5, 9.5, 200)
mh_grid  = np.linspace(-0.8, 0.2, 200)
Age, MH  = np.meshgrid(age_grid, mh_grid)
pos      = np.dstack((Age, MH))
Z        = rv.pdf(pos)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Joint distribution
axes[0].contourf(age_grid, mh_grid, Z, levels=20, cmap="Blues")
axes[0].set_xlabel("log Age (Gyr)")
axes[0].set_ylabel("[Fe/H]")
axes[0].set_title("Joint f(age, [Fe/H])")

# Marginal over [Fe/H] → age distribution
d_mh = mh_grid[1] - mh_grid[0]
f_age = Z.sum(axis=0) * d_mh
axes[1].fill_between(age_grid, f_age, alpha=0.5, color="#3B8BD4")
axes[1].plot(age_grid, f_age, "#3B8BD4", lw=2)
axes[1].set_xlabel("log Age (Gyr)")
axes[1].set_ylabel("Marginal density")
axes[1].set_title("Marginal: f(age)  [after integrating over [Fe/H]]")

# Marginal over age → metallicity distribution
d_age = age_grid[1] - age_grid[0]
f_mh = Z.sum(axis=1) * d_age
axes[2].fill_between(mh_grid, f_mh, alpha=0.5, color="#1D9E75")
axes[2].plot(mh_grid, f_mh, "#1D9E75", lw=2)
axes[2].set_xlabel("[Fe/H]")
axes[2].set_ylabel("Marginal density")
axes[2].set_title("Marginal: f([Fe/H])  [after integrating over age]")

plt.tight_layout()
plt.show()


## 1.5 The Central Limit Theorem

**Theorem:** The mean of n i.i.d. random variables with mean μ and variance σ²
approaches a Gaussian as n → ∞, regardless of the original distribution.

This is why Gaussian noise is ubiquitous — it emerges from sums of many small effects.
But **beware**: at low photon counts (λ < ~20), Poisson ≠ Gaussian.


In [ ]:
# CLT demonstration: sum of Poisson variables
# Show convergence to Gaussian for different n and λ

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
lam = 2.0
n_values = [1, 2, 5, 30]

for col, n in enumerate(n_values):
    # Simulate 50000 means of n Poisson(λ) variables
    samples = np.random.poisson(lam, size=(50000, n)).mean(axis=1)
    mu_clt  = lam
    sig_clt = np.sqrt(lam / n)
    x_range = np.linspace(max(0, mu_clt - 4*sig_clt), mu_clt + 4*sig_clt, 300)
    gauss   = stats.norm.pdf(x_range, mu_clt, sig_clt)

    # Top row: histogram + Gaussian overlay
    ax = axes[0, col]
    ax.hist(samples, bins=40, density=True, color="#3B8BD4", alpha=0.7, label=f"n={n} Poisson means")
    ax.plot(x_range, gauss, "#D85A30", lw=2.5, label="CLT Gaussian")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("Sample mean")
    if col == 0: ax.set_ylabel("Density")
    ax.legend(fontsize=8, frameon=False)

    # Bottom row: QQ plot (how Gaussian is the distribution?)
    ax2 = axes[1, col]
    standardised = (samples - samples.mean()) / samples.std()
    (osm, osr), _ = stats.probplot(standardised, dist="norm")
    ax2.scatter(osm, osr, s=2, alpha=0.3, color="#7F77DD")
    ax2.plot([-4, 4], [-4, 4], "#D85A30", lw=2, ls="--", label="Perfect Gaussian")
    ax2.set_xlim(-4, 4); ax2.set_ylim(-4, 4)
    ax2.set_xlabel("Theoretical quantiles")
    if col == 0: ax2.set_ylabel("Sample quantiles")
    ax2.set_title(f"QQ plot (n={n})")
    ax2.legend(fontsize=8, frameon=False)

axes[0, 0].set_title(f"n = 1 (Poisson itself)
λ = {lam}")
plt.suptitle(f"Central Limit Theorem — Means of Poisson(λ={lam}) Variables", 
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Conclusion: By n=30, the distribution of means is visually indistinguishable")
print("from a Gaussian — even starting from Poisson(λ=2), which is quite skewed.")


## 1.6 Law of Total Probability — A Galaxy Survey Example

**Scenario:** A survey contains galaxies of three morphological types.
We want P(star-forming) = probability a randomly selected galaxy is forming stars.

$$P(\text{SF}) = \sum_i P(\text{SF} \mid \text{type}_i) \cdot P(\text{type}_i)$$


In [ ]:
# Law of Total Probability: galaxy morphology example
types   = ["Spiral", "Elliptical", "Irregular"]
p_type  = np.array([0.30, 0.50, 0.20])     # fraction of each type
p_sf    = np.array([0.80, 0.10, 0.60])     # P(star-forming | type)

p_total_sf = np.sum(p_sf * p_type)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart: P(type) and P(SF|type)
x = np.arange(len(types))
width = 0.35
axes[0].bar(x - width/2, p_type, width, label="P(type)", color="#3B8BD4", alpha=0.85)
axes[0].bar(x + width/2, p_sf,   width, label="P(SF|type)", color="#1D9E75", alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(types)
axes[0].set_ylabel("Probability")
axes[0].set_title("Galaxy Type Fractions and Star-Formation Rates")
axes[0].axhline(p_total_sf, ls="--", color="#D85A30", lw=2,
                label=f"P(SF total) = {p_total_sf:.3f}")
axes[0].legend(frameon=False)

# Stacked contribution from each type
contributions = p_sf * p_type
axes[1].barh(types, contributions,
             color=["#3B8BD4", "#1D9E75", "#D85A30"], alpha=0.85)
axes[1].axvline(p_total_sf, ls="--", color="black", lw=1.5,
                label=f"Total P(SF)={p_total_sf:.3f}")
axes[1].set_xlabel("Contribution to P(SF)")
axes[1].set_title("Each Type's Contribution to Total P(star-forming)")
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()

print("Law of Total Probability:")
for t, pt, psf in zip(types, p_type, p_sf):
    print(f"  P(SF|{t}) × P({t}) = {psf:.2f} × {pt:.2f} = {psf*pt:.4f}")
print(f"  ─────────────────────────────────────")
print(f"  P(SF total)               = {p_total_sf:.4f}")

# Bayes flip: P(Spiral | SF) using Bayes' theorem (preview of Ch. 2)
p_spiral_given_sf = (p_sf[0] * p_type[0]) / p_total_sf
print(f"
Preview of Ch.2: P(Spiral | star-forming) = {p_spiral_given_sf:.3f}")


## 1.7 Summary and Key Formulae

| Distribution | PMF / PDF | Mean | Variance | Astrophysical use |
|---|---|---|---|---|
| Poisson(λ) | λᵏe⁻λ/k! | λ | λ | Photon counts, event rates |
| Gaussian(μ,σ²) | (2πσ²)^(-½) exp(-(x-μ)²/2σ²) | μ | σ² | Measurement noise |
| Cauchy(x₀,γ) | 1/[πγ(1+(x-x₀)²/γ²)] | undefined | undefined | Spectral line shapes |

**Key operations:**
- **Marginalisation:** f_X(x) = ∫ f(x,y) dy — eliminate nuisance variables
- **CLT:** means of large samples → Gaussian, regardless of underlying distribution
- **Total probability:** P(A) = Σ P(A|Bᵢ)P(Bᵢ) — combine conditional and marginal

**Next:** In Chapter 2 we use these foundations to derive Bayes' theorem and
define the full Bayesian inference framework.


In [ ]:
# Final: interactive comparison of all three distributions
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-8, 8, 1000)

ax.plot(x, stats.norm.pdf(x, 0, 1),   "#3B8BD4", lw=2.5, label="Gaussian N(0,1)")
ax.plot(x, stats.cauchy.pdf(x, 0, 1), "#D85A30", lw=2.5, label="Cauchy (Lorentzian)")

# Approximate Poisson(5) as continuous for comparison
k = np.arange(0, 16)
pmf = stats.poisson.pmf(k, 5)
ax.bar(k - 5, pmf, width=0.7, alpha=0.4, color="#1D9E75",
       label="Poisson(λ=5), shifted to 0")

ax.set_xlim(-7, 7)
ax.set_xlabel("x (standardised)")
ax.set_ylabel("Probability density / mass")
ax.set_title("Three Key Distributions in Astrophysical Data Analysis")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print("Ready for Chapter 2: Bayes' Theorem!")
